# Trace the Ace — Runner

**This notebook is a thin, stable wrapper. All logic lives in the `traceace` package on GitHub.**

The loop: *cell errors → I fix it locally → `git push` → you re-run **that one cell***.
`run()` calls `sync()` first, so every cell picks up the latest code **without a runtime reset**.
Results land in `artifacts/` on the local SSD and are pushed to Drive by
`maintenance.sync_artifacts`.

---

## 🚦 Which cells need a GPU — read this before attaching anything

**Units burn while the runtime is _connected_, not while it computes.** An idle attached GPU is
the single biggest waste vector in this project. The tier guard will refuse to run a CPU task on
a GPU, so you cannot waste units by accident — but you can waste them by leaving a runtime up.

| § | Cell | Runtime | Cost |
|---|---|---|---|
| 1–5 | staging, EDA, folds, features, baselines | 🖥️ **CPU + High RAM** | free |
| 6 | **the diagnosis** — `evaluate.by_objective_fold` | 🖥️ **CPU** | free |
| 7 | `evaluate.objective_repeated` — settles `lo_text_difficulty` | 🖥️ **CPU** · ~10 min | free |
| **8** | **`model.transcript_encoder` SMOKE** | ⚡ **L4** · ~10 min | **~1 unit** |
| **9** | **`model.transcript_encoder` 1 fold** | ⚡ **A100** · ~30 min | **~6 units** |
| **10** | **`model.transcript_encoder` all 5 folds** | ⚡ **A100** · ~3–4 h | **~40 units** |
| 11 | blend + calibrate on the encoder OOF | 🖥️ **CPU** | free |
| 12 | build + verify the submission | 🖥️ **CPU** · ~2 min | free |
| **13** | `submission.smoke` timing parity | ⚡ **A100** · ~20 min | **~4 units** |
| 14 | research / rejected experiments | mixed — **off by default** | — |
| 15 | close-out: sync to Drive, budget, disconnect | any | free |

**Only cells 8, 9, 10 and 13 need a GPU.** Everything else must run on CPU + High RAM.
Total GPU plan ≈ **50 units** against ~729 remaining, so the budget is not the constraint —
the 2026-08-27 deadline is.

---

## 📋 Where we actually are (see [`docs/ENDGAME.md`](../docs/ENDGAME.md))

Leaderboard **0.6106 / AUROC 0.6014 / rank #45**. Decomposing the gap to #1:

- **93% of it is discrimination (AUROC), not calibration.** The model is already *optimally*
  calibrated on the test set — the two shrinkage submissions measured `w* = 1.0` directly.
- The per-objective difficulty lookup supplies ~80% of our session-CV AUROC and is worth
  **0.500** on unseen objectives, which is the regime the test set is in.
- So **session-grouped CV has been ranking our models by a quantity the leaderboard does not
  measure**, and several shelved models were shelved against a baseline that does not exist there.

Everything below is judged by **within-objective-fold AUROC**, converted to a projected
leaderboard score by `LB ≈ 0.62393 − 1.321·(AUC−0.5)²`. Targets: **0.622 → top-15**,
0.630 → top-10, 0.645 → #1.


### 🖥️ RUNTIME: **CPU + High RAM** — ~0 units/hr · run once per runtime
Mounts Drive, clones/pulls the repo, defines `sync()` and `run()`.


In [ ]:
import os, sys, subprocess, traceback
from google.colab import drive
drive.mount('/content/drive')

REPO_URL   = "https://github.com/meteorboyF/Trace-the-Ace--meteorboyF.git"
REPO_DIR   = "/content/Trace-the-Ace"
DRIVE_ROOT = "/content/drive/MyDrive/trace-the-ace"
BRANCH     = "main"

# --- GPU gates. Each is False until the cheaper step above it has passed. ---------
RUN_ENCODER_SMOKE = False   # cell 8  · L4   · ~1 unit  · set True with an L4 attached
RUN_ENCODER_FOLD0 = False   # cell 9  · A100 · ~6 units · only after the smoke is green
RUN_ENCODER_FULL  = False   # cell 10 · A100 · ~40 units· only after fold 0 clears the gate
RUN_A100_TIMING   = False   # cell 13 · A100 · ~4 units · only right before a real submission

# --- Research / rejected work. Off by default; see docs/ENDGAME.md and STATE.md. --
RUN_FULL_GPU = False          # frozen BGE window embeddings (already extracted once)
RUN_EXPERIMENTAL_GPU = False  # vLLM move annotation + frozen ModernBERT
RUN_ATTENTION_GPU = False     # supervised BGE attention — REJECTED, kept reachable
RUN_TRANSFORMER_GPU = False   # hierarchical ModernBERT — REJECTED (AUROC 0.4404)
RUN_LEGACY_CPU = False        # session-CV ablations: superseded metric, kept reproducible

# CI asserts this manifest equals BOTH the task registry and the run(...) calls below,
# so a registered task can never become silently unreachable from this notebook.
PIPELINE_TASKS = {
    "annotate.moves", "baseline.lo_only", "baseline.prior", "budget.report",
    "calibrate.fit", "calibrate.shrinkage", "cv.build", "cv.robust_build",
    "data.consolidate", "data.ingest", "docs.build", "eda.inference_budget",
    "eda.lo_conditioning", "eda.overview", "eda.roles", "eda.transcripts",
    "ensemble.blend", "ensemble.promote_text",
    "evaluate.by_objective_fold", "evaluate.objective_noise_floor",
    "evaluate.objective_repeated", "evaluate.repeated", "evaluate.report",
    "evaluate.robust_gbdt", "evaluate.semantic_repeated", "evaluate.unseen_lo",
    "features.content", "features.embeddings", "features.feedback",
    "features.linguistic", "features.lo_alignment", "features.structural",
    "features.temporal", "features.trajectory", "features.window_embeddings",
    "interpret.ablation", "interpret.ablation_repeated", "interpret.report",
    "maintenance.restore_artifacts", "maintenance.sync_artifacts",
    "model.bge_attention", "model.gbdt", "model.hierarchical_transformer",
    "model.move_classifier", "model.sparse_text", "model.transcript_encoder",
    "selftest.all", "submission.build", "submission.smoke", "submission.verify",
}

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

def sync(branch=BRANCH, install=False):
    """git pull -> purge cached modules -> re-import. Returns the package."""
    subprocess.run(["git","-C",REPO_DIR,"fetch","--all","--quiet"], check=True)
    subprocess.run(["git","-C",REPO_DIR,"checkout",branch,"--quiet"], check=True)
    subprocess.run(["git","-C",REPO_DIR,"reset","--hard",f"origin/{branch}","--quiet"], check=True)
    src = os.path.join(REPO_DIR, "src")
    if src not in sys.path:
        sys.path.insert(0, src)
    if install:
        subprocess.run([sys.executable,"-m","pip","install","-q","-r",
                        os.path.join(REPO_DIR,"requirements-colab.txt")], check=True)
    for m in [m for m in list(sys.modules) if m == "traceace" or m.startswith("traceace.")]:
        del sys.modules[m]
    import traceace
    traceace.configure(repo_dir=REPO_DIR, drive_root=DRIVE_ROOT)
    sha = subprocess.run(["git","-C",REPO_DIR,"rev-parse","--short","HEAD"],
                         capture_output=True, text=True).stdout.strip()
    print(f"synced @ {sha}")
    return traceace

def run(task, **kw):
    """Sync, then run one task; failures stop the cell and all dependent work."""
    tc = sync()
    try:
        return tc.tasks.run(task, **kw)
    except Exception:
        traceback.print_exc()
        raise

tc = sync(install=True)   # install=True only needed once per runtime


---
## 1 · Data staging and EDA

### 🖥️ RUNTIME: **CPU + High RAM** — ~0 units/hr · est. 5–10 min (first run)
Stages `raw.zip` from Drive to the local SSD and extracts it **locally** (Drive is a FUSE mount
at ~100–300 ms per file op — never iterate it). No-op if already staged.
`maintenance.restore_artifacts` recovers previously-paid caches (BGE vectors, feature blocks)
from Drive so a fresh runtime does not re-earn them.

Do **not** run on GPU.


In [ ]:
run("data.ingest")                    # normalize suffixed filenames by CONTENT SHAPE
run("maintenance.restore_artifacts")  # recover paid caches on a fresh runtime
run("data.consolidate")               # 22,821 transcript CSVs -> one parquet (once)


### 🖥️ RUNTIME: **CPU + High RAM** — ~0 units/hr · est. 8 min · **skip if `docs/DATA.md` is current**
Measurement only — these numbers are already recorded and drive nothing new. Re-run after any
change to ingestion.


In [ ]:
run("eda.overview")
run("eda.roles")
run("eda.lo_conditioning")
run("eda.transcripts")          # exact token/char distributions (~8 min, progress bar)
run("eda.inference_budget")     # go/no-go vs the 6-hour A100 cap


---
## 2 · Folds and baselines

### 🖥️ RUNTIME: **CPU + High RAM** — ~0 units/hr · est. <1 min

Two fold systems, and the distinction now decides everything:

- **`cv.build`** — session-grouped. Necessary (one session → many responses) but **not
  sufficient**: it lets the same learning objectives appear on both sides, which is how a
  per-objective lookup scores 0.706 in CV and 0.500 on the leaderboard.
- **`cv.robust_build(kind="objective")`** — whole objectives held out, with validation sessions
  *and* objectives purged from the training side. **This is the regime the test set is in.**

`baseline.lo_only` remains the organisers' anti-goal implemented as a baseline.


In [ ]:
run("cv.build")
run("cv.robust_build", kind="objective")   # THE promotion folds
run("baseline.prior")       # the floor  (expected logloss ~0.609)
run("baseline.lo_only")     # THE BAR    (no transcript information at all)


---
## 3 · Feature blocks  (all CPU — the cheap ladder)

### 🖥️ RUNTIME: **CPU + High RAM** — ~0 units/hr · est. 20 min total
Cached to parquet; reruns are free (loud `CACHE HIT`). Pass `force=True` to recompute.
`features.lo_alignment` is built because feedback, trajectory **and the neural encoder** are all
scoped to the sliding windows it defines.


In [ ]:
run("features.structural")      # ~2.5 min
run("features.linguistic")      # ~6 min  (incl. ASR disfluency markers)
run("features.temporal")        # ~3 min  (robust statistics)
run("features.lo_alignment")    # windows used by every response-level block
run("features.feedback")
run("features.trajectory")      # strongest block under session CV
run("cv.robust_build", kind="domain")  # transcript-style holdouts need the CPU blocks


---
## 4 · The GBDT baseline

### 🖥️ RUNTIME: **CPU + High RAM** — ~0 units/hr · est. 5 min
The currently-shipped model, retrained so the OOF on disk matches the code on disk. Its session-CV
score (~0.543) is **not** a leaderboard predictor — see §6.


In [ ]:
run("model.gbdt")
run("calibrate.fit", experiment="model.gbdt")
run("evaluate.report", experiment="model.gbdt")
run("interpret.report", experiment="model.gbdt")   # figures -> artifacts/figures/


---
## 5 · 🔍 THE DIAGNOSIS — run this before deciding anything

### 🖥️ RUNTIME: **CPU + High RAM** — ~0 units/hr · est. <1 min

Re-reads **every** experiment on disk through the metric that actually tracks the leaderboard:
within-held-out-objective AUROC, plus a projected LB score.

**How to read the output.**
- Rows marked `honest` were trained objective-disjoint. **Only these are promotion evidence.**
- Rows marked `optimistic` were trained on session folds and saw their validation objectives.
  `model.gbdt` will appear near the top at ~0.72 — that number is an artifact, and reproducing
  it is exactly the mistake that put us at #45.
- `constant_predictor_pooled_auc` should be ~0.457, not 0.500. That is the pooling trap this
  harness exists to avoid; it is reported so it stays visible.

Expect the honest best to sit near **0.596 → projected LB 0.6117**, against our real 0.6106.


In [ ]:
run("evaluate.by_objective_fold")   # sweeps every OOF on disk


---
## 6 · The promotion gate — where every A/B gets settled

### 🖥️ RUNTIME: **CPU + High RAM** — ~0 units/hr · est. 10–15 min

**Objective folds are brutally noisy.** Each holds only ~80 of 398 objectives and objective
difficulty is lumpy, so a single assignment gives paired deltas with an SD around 0.037 — larger
than any effect we are chasing. This harness repeats the comparison across **5 different
assignments of the same objectives** and reports a paired mean with a real standard error.
Promotion requires the mean to clear **2× its standard error** *and* a consistent sign.

`evaluate.objective_noise_floor` runs the harness with **identical** arms, so whatever spread it
reports is pure noise. Run it once — it calibrates every A/B that follows.

> ### ❌ Already settled: objective-text difficulty is **rejected**
> The per-objective lookup dies on unseen objectives (AUC 0.500), and objective *descriptions*
> do carry standalone signal — a text→difficulty model scores **0.575** on objectives it has
> never seen. On one fold assignment adding it looked like **+0.0105 AUROC**.
>
> Across five assignments it is **−0.00478 ± 0.00577, positive in 1/5**. The single-assignment
> gain was noise, and the sign flipped. `include_lo_text_difficulty` stays **False** and nothing
> ships. Re-run the cell to reproduce; this is a negative result for the write-up, and a direct
> measurement of how little objective metadata transfers — the organisers' anti-goal, quantified.

Use this harness for the next question instead: whether the encoder's OOF adds to the GBDT.


In [ ]:
run("evaluate.objective_noise_floor")   # what does zero effect look like? (~10 min)
run("evaluate.objective_repeated", label="lo_text_difficulty")   # reproduces the rejection


---
## 7 · ⚡ The neural transcript model — SMOKE

### ⚡ RUNTIME: **L4 GPU** — ~5 units/hr · est. ~10 min ≈ **1 unit**

This is the bet. 93% of the gap is discrimination, hand-crafted features have plateaued at
AUROC ~0.606, and the leaders sit at 0.63–0.64 — which is what a competent transformer fine-tune
on tutoring dialogue looks like.

**The design, and why it differs from the pilot that failed at AUROC 0.4404:**
- **One example per response**, not per session — 26% of outcome variance is *between objectives
  inside one session*, invisible to any session-level representation.
- **Retrieval, not truncation** — the top-4 windows most relevant to *this* objective, selected by
  the same `topk_spans` the submission uses, at 2,048 tokens (94% of examples fit intact).
- **A flat encoder and one head.** No dual heads, no conditional blending.
- **`include_objective=False` by default** — the encoder reads *dialogue only*. Handing it the
  objective text invites exactly the memorisation that scores 0.706 in CV and 0.500 on the
  leaderboard, and it is the organisers' stated anti-goal.

**Set `RUN_ENCODER_SMOKE = True` in the setup cell, re-run setup, then run this.**
Gate: it must complete and report a finite AUC. Errors here are free to fix — push and re-run.


In [ ]:
if RUN_ENCODER_SMOKE:
    # gradient_checkpointing=True: the L4 has 24 GB and batch 8 x 2048 tokens with
    # gradients does not reliably fit without it. The A100 cells leave it off for speed.
    run("model.transcript_encoder",
        subsample=400, folds=0, split_mode="session",
        experiment="smoke.transcript_encoder", epochs=1,
        gradient_checkpointing=True)
else:
    print("Smoke disabled. Attach an L4, set RUN_ENCODER_SMOKE=True, re-run setup, re-run this.")


### ⚡ RUNTIME: **A100 80GB** — ~12 units/hr · est. ~30 min ≈ **6 units**

One honest fold on the full data, objective-disjoint. **This is the go/no-go.**

| fold-0 AUROC | verdict |
|---|---|
| **≥ 0.60** | promising — spend the 40 units on all five folds |
| 0.55 – 0.60 | comparable to the GBDT; try `max_tokens=3072, topk_windows=6` or `epochs=3` first |
| < 0.55 | something is wrong. Do **not** spend the 40 units — push a fix and re-run this cell |

Fold predictions are written the moment the fold finishes, so a disconnect costs one fold, not
the run. Re-running skips folds already on disk.

**Set `RUN_ENCODER_FOLD0 = True`, re-run setup, then run this.**


In [ ]:
if RUN_ENCODER_FOLD0:
    run("model.transcript_encoder", folds=0, split_mode="objective", allow_waste=True)
else:
    print("Fold 0 disabled. Pass the smoke first, then set RUN_ENCODER_FOLD0=True.")


### ⚡ RUNTIME: **A100 80GB** — ~12 units/hr · est. 3–4 h ≈ **40 units**

All five objective-disjoint folds, producing the OOF that everything downstream blends on. Fold 0
is already on disk from the previous cell and will be skipped.

`shutdown_after=True` stops billing the moment it finishes, so this is safe to leave overnight.

**Set `RUN_ENCODER_FULL = True`, re-run setup, then run this.** Only after fold 0 cleared 0.60.


In [ ]:
if RUN_ENCODER_FULL:
    run("model.transcript_encoder", split_mode="objective",
        allow_waste=True, shutdown_after=True)
else:
    print("Full run disabled. Clear the fold-0 gate first, then set RUN_ENCODER_FULL=True.")


---
## 8 · Blend, calibrate, and re-diagnose

### 🖥️ RUNTIME: **CPU + High RAM** — ~0 units/hr · est. 10 min

**Why not blend with the shipped `model.gbdt` directly?** Its OOF was trained on session folds,
so it carries memorized per-objective difficulty — AUC 0.72 in CV, ~0.60 on the leaderboard.
Weights fitted against that OOF reward a signal the test set does not have, and the encoder
would be underweighted for exactly the wrong reason. `ensemble.blend` now warns loudly on
mixed regimes and records `input_split_modes` in its result.

So this cell trains an **honest GBDT twin** (objective-disjoint, transcript-only) and blends the
two honest OOFs. The resulting weight is the dialogue-features-vs-encoder mix that actually
transfers; the deployed model applies that weight on top of the full GBDT (whose lo_prior is
~free on seen objectives and inert on unseen ones).

Read `evaluate.by_objective_fold` afterwards: the blend must appear as `honest` and beat
**0.622** to reach the top-15 write-up gate.


In [ ]:
# BOTH blend inputs must be honest (objective-disjoint), or the weight optimizer loads
# onto memorized objective difficulty that does not exist on the test set. So: first an
# honest GBDT twin, then the blend, then the same diagnosis as everything else.
run("model.gbdt", split_mode="objective", include_lo_prior=False,
    experiment="model.gbdt_objective")
run("ensemble.blend", experiments=["model.gbdt_objective", "model.transcript_encoder"],
    output_experiment="ensemble.dialogue")
run("calibrate.fit", experiment="ensemble.dialogue")
run("evaluate.by_objective_fold")     # did it actually move? honest rows only
run("evaluate.repeated")              # session-CV regression guard, not a promotion metric


---
## 9 · Submission

### 🖥️ RUNTIME: **CPU + High RAM** — ~0 units/hr · est. 2 min

`verify` is deliberately paranoid: **three full submissions per rolling week**, and ~4–5 remain
before 2026-08-27. A loud verify is worth more than a clever model — submission #1 scored *below
random* because a feature-order bug shipped silently, and ADR-018 records nine output checks that
never executed while the report still looked green.

⚠️ **Inspect the full verifier report before uploading.** Never upload on a green summary line.


In [ ]:
run("selftest.all")                     # full behavioural gate before packaging
run("submission.build", experiment="model.gbdt")
run("submission.verify", smoke=True)    # raises on ANY violation
run("submission.smoke")                 # runs main.py as a subprocess (CPU parity)


### ⚡ RUNTIME: **A100 80GB** — ~12 units/hr · est. 20 min ≈ **4 units**
**Only for final timing validation** against the real inference hardware, immediately before a
real submission. Nothing else needs an A100.


In [ ]:
if RUN_A100_TIMING:
    run("submission.smoke", allow_waste=True)   # A100 timing parity check
else:
    print("A100 timing check disabled — enable only right before a real upload.")


---
## 10 · Research and rejected experiments

### 🖥️/⚡ RUNTIME: **mixed — all gated off by default**

Kept reachable so every registered task stays exercised and every negative result stays
reproducible. **Nothing here is on the critical path to 2026-08-27.**

- `model.sparse_text`, `ensemble.promote_text` — deployable sparse dialogue model.
- `annotate.moves` + `model.move_classifier` — the tutoring-move taxonomy. This is the
  **write-up centrepiece** (an organiser-named research direction) and the Phase-2 fallback if the
  encoder underdelivers. vLLM backend is dev-time only: no generative model enters the submission,
  and competition data may never be sent to a hosted API.
- `features.window_embeddings` / `features.content` / `model.bge_attention` — frozen BGE work.
  Already extracted once (4 units); the attention model was **rejected**.
- `model.hierarchical_transformer` — **rejected** (AUROC 0.4404, below random). Superseded by
  `model.transcript_encoder`.
- Session-CV ablations — superseded metric, retained for the write-up's negative results.


In [ ]:
run("model.sparse_text")
run("ensemble.promote_text")      # text enters only after fold-held-out gain
run("annotate.moves", backend="heuristic")
run("model.move_classifier")

if RUN_LEGACY_CPU:
    run("interpret.ablation")
    run("interpret.ablation_repeated")     # session-CV metric — superseded, see ENDGAME.md
    run("evaluate.unseen_lo")
    run("evaluate.robust_gbdt", kind="objective")
    run("evaluate.robust_gbdt", kind="domain")
    run("evaluate.semantic_repeated")
    run("features.lo_alignment", backend="embedding")
    run("features.content")

if RUN_EXPERIMENTAL_GPU:
    run("annotate.moves", backend="vllm", subsample=200)  # research smoke (~40-unit full job)
    run("features.embeddings", subsample=500)             # frozen ModernBERT; may OOM on L4
if RUN_FULL_GPU:
    run("features.window_embeddings", subsample=500)      # BGE smoke
    run("features.window_embeddings")                     # full extraction (already paid once)
if RUN_ATTENTION_GPU:
    run("model.bge_attention")            # REJECTED: 0.00717 worse than objective-only
if RUN_TRANSFORMER_GPU:
    run("model.hierarchical_transformer", subsample=500, split_mode="objective", fold=0)

if False:  # RETIRED: leaderboard log loss worsened 0.6106 -> 0.6133 at identical AUROC
    run("calibrate.shrinkage", experiment="model.gbdt")


---
## 11 · Session close-out

### 🖥️ RUNTIME: **any** — run this at the END of every session

Syncs artifacts to Drive (tarred — one big file, never a directory walk), prints the budget
report, regenerates `docs/EXPERIMENTS.md`, then disconnects so billing stops.

**Uncomment the last two lines if a paid GPU is attached.**


In [ ]:
run("maintenance.sync_artifacts")   # tars artifacts + runs, single large write to Drive
run("budget.report")                # spend by task and tier vs the 733-unit balance
run("docs.build")                   # regenerate docs/EXPERIMENTS.md from runs/

# Disconnect so an attached GPU stops burning units:
# from google.colab import runtime; runtime.unassign()
